# Cell Ranger Multi - QC Summary

Aggregate and visualize Cell Ranger metrics across all samples.
Run after Cell Ranger multi has completed for all samples.

**Checks:**
- Estimated cells recovered (flag <1,000)
- Median genes per cell (flag <500)
- Sequencing saturation (flag <50%)
- VDJ metrics: productive chain pairing rate

In [ ]:
import sys
sys.path.insert(0, '../pipeline')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from utils import load_config, load_metadata, get_gex_samples, set_plotting_defaults
set_plotting_defaults()

cfg = load_config('../pipeline/config.yaml')
meta = load_metadata(cfg)

In [ ]:
# Run the verification script to aggregate metrics
from importlib import import_module
cr_module = import_module('00_cellranger_multi')

verification = cr_module.verify_cellranger_outputs(cfg, meta)
print(f"Samples checked: {len(verification)}")
print(f"Status counts:")
print(verification['status'].value_counts())
verification.head(10)

In [ ]:
# Plot cells recovered per sample
ok_data = verification[verification['cells'].notna()].copy()
ok_data = ok_data.sort_values('cells')

fig, axes = plt.subplots(2, 1, figsize=(max(14, len(ok_data) * 0.4), 10))

# Cells recovered
colors = ['firebrick' if c < 1000 else 'steelblue' for c in ok_data['cells']]
axes[0].bar(range(len(ok_data)), ok_data['cells'], color=colors)
axes[0].axhline(1000, ls='--', color='red', alpha=0.5, label='Threshold (1000)')
axes[0].axhline(3000, ls='--', color='green', alpha=0.5, label='Expected (3000)')
axes[0].set_ylabel('Estimated cells')
axes[0].set_title('Cells recovered per sample')
axes[0].legend()
axes[0].set_xticks(range(len(ok_data)))
axes[0].set_xticklabels(ok_data['sample'], rotation=90, fontsize=7)

# Median genes
if 'median_genes' in ok_data.columns:
    ok_genes = ok_data[ok_data['median_genes'].notna()].sort_values('median_genes')
    colors = ['firebrick' if g < 500 else 'steelblue' for g in ok_genes['median_genes']]
    axes[1].bar(range(len(ok_genes)), ok_genes['median_genes'], color=colors)
    axes[1].axhline(500, ls='--', color='red', alpha=0.5, label='Threshold (500)')
    axes[1].set_ylabel('Median genes per cell')
    axes[1].set_title('Median genes per cell per sample')
    axes[1].legend()
    axes[1].set_xticks(range(len(ok_genes)))
    axes[1].set_xticklabels(ok_genes['sample'], rotation=90, fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Flagged samples
flagged = verification[verification['status'] == 'FLAG']
if len(flagged) > 0:
    print("FLAGGED SAMPLES:")
    print(flagged[['sample', 'flags', 'cells', 'median_genes', 'saturation']].to_string())
else:
    print("No flagged samples - all metrics within acceptable range.")

# Missing samples
missing = verification[verification['status'] == 'MISSING']
if len(missing) > 0:
    print(f"\nMISSING SAMPLES ({len(missing)}):")
    print(missing['sample'].tolist())